# 05 Ekstraksi Kata Kunci Tanpa Label

Mengambil kata atau frasa penting tanpa data latih. Metode dinilai pada KEPS yang punya jawaban, lalu
diterapkan ke komentar MBG.

**Daftar isi**

1. Kartu Masalah
2. Kapan Pakai Notebook Ini
3. Config
4. EDA
5. Preprocessing
6. Validasi
7. Baseline
8. Model Utama
9. Post-processing
10. Post-analysis
11. Keluaran

## 1. Kartu Masalah

| Bagian | Isi |
|---|---|
| **Task** | Mengambil kata kunci tanpa label |
| **Unit prediksi** | Satu dokumen mendapat beberapa frasa teratas |
| **Metrik** | Precision, recall, dan F1 pada KEPS, dengan pencocokan persis |
| **Masalah terbesar** | Rata-rata hanya 4 frasa per dokumen dan pencocokannya harus persis, jadi skor absolut memang rendah. Yang bermakna adalah perbandingan antar metode. |
| **Data contoh** | KEPS untuk uji, komentar MBG untuk demo |

## 2. Kapan Pakai Notebook Ini

**Pakai kalau** soal meminta kata kunci, isu utama, atau frasa penting, dan tidak ada label. Kalau ada label
frasa kunci, pakai folder `18`.

**Yang perlu diganti kalau datanya beda:**
- `load_target` dan `TEXT_COL`.
- `DOMAIN_STOPWORDS`.
- Bagian evaluasi KEPS boleh dibiarkan, karena itu acuan pemilihan metode untuk bahasa Indonesia.

## 3. Config

**Import dan lingkungan.** Mode offline dinyalakan di sini, dan progress bar serta warning
dimatikan supaya output tetap bersih.

In [1]:
import importlib.util
import json
import os
import random
import subprocess
import sys
import time
import warnings
from contextlib import contextmanager
from pathlib import Path

os.environ.update(
    CUDA_VISIBLE_DEVICES="0",
    HF_HUB_OFFLINE="1",
    TRANSFORMERS_OFFLINE="1",
    TOKENIZERS_PARALLELISM="false",
    TQDM_DISABLE="1",
    PYTHONWARNINGS="ignore",
    HF_HUB_DISABLE_PROGRESS_BARS="1",
)
warnings.filterwarnings("ignore")
sys.modules["tensorflow"] = None  # tidak dipakai; mencegah log TensorFlow saat umap diimpor

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
from transformers.utils import logging as hf_logging  # noqa: E402

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.max_colwidth", 90, "display.max_rows", 30, "display.width", 140)
plt.rcParams.update(
    {
        "figure.figsize": (8, 3.5),
        "figure.dpi": 90,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
print(f"device={DEVICE} | torch={torch.__version__}")

device=cuda | torch=2.11.0+cu130


**Lokasi file.** `find_pack` mencari paket offline, dan `find_file` mencari file data di Kaggle
atau folder lokal. Di mesin lokal, set `PACK_DIR` dan `DATA_ROOT` lewat environment variable.

In [2]:
INPUT_ROOTS = [Path(p) for p in (os.environ.get("DATA_ROOT"), "/kaggle/input", ".") if p]


def find_pack() -> Path:
    """Cari folder paket offline lewat file pack_manifest.json."""
    if os.environ.get("PACK_DIR"):
        return Path(os.environ["PACK_DIR"])
    for root in INPUT_ROOTS:
        for manifest in root.glob("**/pack_manifest.json"):
            return manifest.parent
    raise FileNotFoundError("Paket offline tidak ketemu. Set environment variable PACK_DIR.")


def find_file(name: str) -> Path:
    """Cari file data berdasarkan nama di semua folder input."""
    for root in INPUT_ROOTS:
        for path in root.glob(f"**/{name}"):
            return path
    raise FileNotFoundError(f"File {name} tidak ketemu di {INPUT_ROOTS}")


def install_offline(*modules: str) -> None:
    """Pasang library dari wheel paket offline, hanya bila belum terpasang."""
    missing = [m for m in modules if importlib.util.find_spec(m.replace("-", "_")) is None]
    if missing:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-index",
                "--find-links",
                str(PACK_DIR / "wheels"),
                "--root-user-action=ignore",
                *missing,
            ],
            check=True,
        )


PACK_DIR = find_pack()
MODEL_DIR = PACK_DIR / "models"
EXTRA_DATA_DIR = PACK_DIR / "data"
RESULT_DIR = Path("hasil")
RESULT_DIR.mkdir(exist_ok=True)
print(f"paket offline: {PACK_DIR}")

paket offline: /kaggle/input/notebooks/ivanwllm/gemastik-nlp-00-setup-offline


**Pencatat waktu dan ringkasan.** `stage` mencatat durasi tiap tahap, dan `run_summary` mencetak
ringkasan akhir yang dibaca skrip audit.

In [3]:
RUN_START = time.time()
STAGE_SECONDS: dict[str, float] = {}


@contextmanager
def stage(name: str):
    """Catat durasi satu tahap."""
    start = time.time()
    yield
    STAGE_SECONDS[name] = round(time.time() - start, 1)
    print(f"[waktu] {name}: {STAGE_SECONDS[name]:.0f} detik")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def run_summary(task: str, n_rows: int, metrics: dict, needs_gpu: bool = True) -> None:
    """Simpan dan cetak ringkasan run."""
    total = int(time.time() - RUN_START)
    used_gpu = torch.cuda.is_available() and torch.cuda.max_memory_allocated() > 0
    summary = {
        "task": task,
        "mode": MODE,
        "use_pretrained": USE_PRETRAINED,
        "n_rows": n_rows,
        "total_detik": total,
        "gpu": used_gpu,
        "tahap_detik": STAGE_SECONDS,
        "metrik": metrics,
    }
    (RESULT_DIR / "ringkasan.json").write_text(json.dumps(summary, indent=1))
    note = "" if needs_gpu else " gpu_tidak_wajib"
    print(
        f"RINGKASAN_RUN task={task} mode={MODE} n_rows={n_rows} total_detik={total} "
        f"gpu={used_gpu}{note}"
    )
    print(json.dumps(metrics, indent=1))

**Pengaturan utama.** Semua yang perlu diubah ada di sel ini.

In [4]:
MODE = "penuh"  # "cepat" (<30 menit) atau "penuh" (<2 jam)
USE_PRETRAINED = True  # False: hanya TF-IDF dan YAKE, tanpa file model dari luar
SEED = 42
N_ROWS = 5000
TOP_K = 5
TEXT_COL = "Comment"
DOMAIN_STOPWORDS = {"mbg", "makan", "bergizi", "gratis", "program"}
MMR_DIVERSITY = 0.5

SETTINGS = {"cepat": {"eval_docs": 400}, "penuh": {"eval_docs": 1000}}[MODE]

seed_everything(SEED)
install_offline("yake")
print(SETTINGS)

{'eval_docs': 1000}


**Util teks.** `clean_text` hanya membuang gangguan teknis (URL, mention, spasi) dan
mempertahankan emoji serta tanda seru karena keduanya membawa sinyal. `normalize_slang` memakai
kamus alay. `near_duplicate_groups` menemukan komentar kembar supaya tidak bocor antar fold.

In [5]:
import re  # noqa: E402

from sklearn.feature_extraction.text import TfidfVectorizer  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import roc_auc_score  # noqa: E402
from sklearn.model_selection import cross_val_predict  # noqa: E402
from sklearn.neighbors import NearestNeighbors  # noqa: E402

URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
SPACE_RE = re.compile(r"\s+")


def clean_text(text: str) -> str:
    """Bersihkan URL, mention, dan spasi berlebih. Huruf kecil, emoji tetap."""
    text = URL_RE.sub(" ", str(text))
    text = MENTION_RE.sub(" ", text)
    return SPACE_RE.sub(" ", text).strip().lower()


def load_slang_dict() -> dict[str, str]:
    """Muat kamus kata informal ke kata baku."""
    path = next((EXTRA_DATA_DIR / "kamus_alay").glob("**/colloquial-indonesian-lexicon.csv"))
    lexicon = pd.read_csv(path)
    return dict(zip(lexicon["slang"].astype(str), lexicon["formal"].astype(str)))


def normalize_slang(text: str, slang: dict[str, str]) -> str:
    """Ganti kata informal dengan kata baku, per kata."""
    return " ".join(slang.get(word, word) for word in text.split())


def near_duplicate_groups(texts: list[str], threshold: float = 0.9) -> np.ndarray:
    """Beri id grup yang sama pada teks yang kemiripan char TF-IDF-nya di atas threshold."""
    vectors = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1).fit_transform(texts)
    neighbors = NearestNeighbors(n_neighbors=min(6, len(texts)), metric="cosine").fit(vectors)
    distance, index = neighbors.kneighbors(vectors)
    parent = list(range(len(texts)))

    def root(i: int) -> int:
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for i, (row_dist, row_idx) in enumerate(zip(distance, index)):
        for dist, j in zip(row_dist, row_idx):
            if dist <= 1 - threshold:
                parent[root(i)] = root(int(j))
    return np.array([root(i) for i in range(len(texts))])


def adversarial_auc(texts_a: list[str], texts_b: list[str]) -> float:
    """AUC model yang membedakan dua kumpulan teks. Dekat 0,5 berarti tidak ada shift."""
    texts = list(texts_a) + list(texts_b)
    target = np.array([0] * len(texts_a) + [1] * len(texts_b))
    vectors = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit_transform(texts)
    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    scores = cross_val_predict(model, vectors, target, cv=3, method="predict_proba")[:, 1]
    return float(roc_auc_score(target, scores))


def text_profile(texts: pd.Series) -> pd.Series:
    """Ringkasan kualitas teks untuk EDA."""
    words = texts.astype(str).str.split().str.len()
    return pd.Series(
        {
            "baris": len(texts),
            "kosong": int((words == 0).sum()),
            "duplikat_persis_%": round(100 * texts.duplicated().mean(), 1),
            "kata_median": float(words.median()),
            "kata_p95": float(words.quantile(0.95)),
            "kata_maks": int(words.max()),
            "ada_url_%": round(100 * texts.astype(str).str.contains(r"https?://").mean(), 1),
            "ada_emoji_%": round(
                100 * texts.astype(str).str.contains(r"[\U0001F300-\U0001FAFF]").mean(), 1
            ),
        }
    )

**Stopword.** Daftar ini hanya dipakai untuk merangkum kata kunci topik, bukan untuk input model
embedding. Tambahkan kata yang terlalu umum di kasus lu, misalnya `mbg` dan `makan`.

In [6]:
STOPWORDS = set(
    """
    ada adalah agar akan aku anda apa apakah atas atau bagaimana bagi bahkan bahwa banyak baru
    begitu belum benar berapa bersama bisa buat bukan cuma dalam dan dapat dari dengan di dia
    dong dulu gak ga gitu gue gua hanya harus hal ini itu iya jadi jangan juga jika kalau kalo
    kami kamu kan karena kata ke kita kok lagi lah lain lalu lebih lu lo masih mau maka mana
    memang mereka meski mungkin nah nanti ni nya oleh pada pak para pun punya saat saja sama
    sampai saya sebagai sebagainya sedang sudah sudah sih seperti siapa supaya tak tapi telah
    tentang terus tetapi tidak tuh udah untuk ya yaitu yang yg dgn dg krn utk tdk jg aja kalo
    aja nih deh kak bang bu ibu bapak sekarang semua setiap sangat tanpa ketika orang enggak
    quot amp mas mbak sih dah tau kayak kaya sampe gimana emang kenapa
    """.split()
)

**Embedding teks.** Kalau pretrained boleh dipakai, teks diubah dengan multilingual-e5. Kalau
tidak boleh, `USE_PRETRAINED=False` memakai TF-IDF + SVD sebagai gantinya.

In [7]:
from sklearn.decomposition import TruncatedSVD  # noqa: E402
from sklearn.feature_extraction.text import TfidfVectorizer as _Tfidf  # noqa: E402
from sklearn.preprocessing import normalize as _l2  # noqa: E402
from transformers import AutoModel, AutoTokenizer  # noqa: E402


def tfidf_svd_embed(texts: list[str], dim: int = 256) -> np.ndarray:
    """Embedding tanpa pretrained: TF-IDF kata + SVD."""
    matrix = _Tfidf(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit_transform(texts)
    dim = min(dim, matrix.shape[1] - 1, matrix.shape[0] - 1)
    return _l2(TruncatedSVD(dim, random_state=SEED).fit_transform(matrix))


@torch.inference_mode()
def e5_embed(texts: list[str], prefix: str = "query: ", batch_size: int = 64) -> np.ndarray:
    """Mean pooling multilingual-e5, hasil dinormalisasi L2."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR / "e5")
    model = AutoModel.from_pretrained(MODEL_DIR / "e5").to(DEVICE).eval().half()
    order = np.argsort([len(t) for t in texts])
    chunks = []
    for start in range(0, len(texts), batch_size):
        batch = [prefix + texts[i] for i in order[start : start + batch_size]]
        tokens = tokenizer(
            batch, padding=True, truncation=True, max_length=128, return_tensors="pt"
        ).to(DEVICE)
        hidden = model(**tokens).last_hidden_state
        mask = tokens["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(1) / mask.sum(1)
        chunks.append(torch.nn.functional.normalize(pooled, dim=1).float().cpu().numpy())
    del model
    torch.cuda.empty_cache()
    vectors = np.empty((len(texts), chunks[0].shape[1]), dtype=np.float32)
    vectors[order] = np.vstack(chunks)
    return vectors


def embed_texts(texts: list[str]) -> np.ndarray:
    """Pilih embedding sesuai USE_PRETRAINED."""
    return e5_embed(texts) if USE_PRETRAINED else tfidf_svd_embed(texts)

## 4. EDA

Muat KEPS sebagai acuan, dan komentar MBG sebagai target. KEPS diberi label frasa dengan pola BIO.

In [8]:
def read_keps(split: str) -> list[tuple[list[str], list[str]]]:
    path = (
        EXTRA_DATA_DIR / "indonlu/dataset/keps_keyword-extraction-prosa" / f"{split}_preprocess.txt"
    )
    sentences, tokens, tags = [], [], []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            if tokens:
                sentences.append((tokens, tags))
            tokens, tags = [], []
            continue
        word, tag = line.split("\t")
        tokens.append(word)
        tags.append(tag)
    return sentences + ([(tokens, tags)] if tokens else [])


def gold_phrases(tokens: list[str], tags: list[str]) -> set[str]:
    phrases, current = set(), []
    for word, tag in zip(tokens, tags):
        if tag == "B":
            if current:
                phrases.add(" ".join(current).lower())
            current = [word]
        elif tag == "I" and current:
            current.append(word)
        else:
            if current:
                phrases.add(" ".join(current).lower())
            current = []
    if current:
        phrases.add(" ".join(current).lower())
    return phrases


keps = (read_keps("train") + read_keps("valid") + read_keps("test"))[: SETTINGS["eval_docs"]]
eval_texts = [" ".join(tokens).lower() for tokens, _ in keps]
eval_gold = [gold_phrases(tokens, tags) for tokens, tags in keps]
target = pd.read_csv(find_file("comment_data.csv")).dropna(subset=[TEXT_COL]).head(N_ROWS)
print(f"dokumen KEPS={len(keps)} | komentar={len(target)}")
print("contoh frasa KEPS:", list(eval_gold[0])[:6])
print("rata-rata frasa per dokumen:", round(np.mean([len(g) for g in eval_gold]), 2))
text_profile(target[TEXT_COL].astype(str))

dokumen KEPS=1000 | komentar=5000
contoh frasa KEPS: ['tutup', 'kartu kredit bca', 'proses telepon', 'melalui', 'ribet']
rata-rata frasa per dokumen: 4.02


baris                5000.0
kosong                  0.0
duplikat_persis_%       0.8
kata_median            17.0
kata_p95               87.0
kata_maks            1446.0
ada_url_%               2.0
ada_emoji_%            13.4
dtype: float64

**Temuan dan keputusan**

| Temuan | Keputusan |
|---|---|
| Frasa KEPS banyak yang satu kata, misalnya `ribet` | Kandidat dibuat dari n-gram 1 sampai 3 kata |
| Frasa teratas sering tumpang tindih | Coba buang tumpang tindih dan MMR, lalu ukur efeknya |
| Komentar MBG punya median 17 kata dan banyak kata umum | Kandidat tidak boleh diawali atau diakhiri stopword |

## 5. Preprocessing

Kandidat frasa berupa n-gram 1 sampai 3 kata yang tidak diawali atau diakhiri stopword. Frasa diambil dari
teks huruf kecil, dan tanda baca memutus frasa.

In [9]:
import html  # noqa: E402

all_stopwords = STOPWORDS | DOMAIN_STOPWORDS
HTML_RE = re.compile(r"<[^>]+>")
target["clean"] = (
    target[TEXT_COL].astype(str).map(lambda t: clean_text(HTML_RE.sub(" ", html.unescape(t))))
)


def candidate_phrases(text: str, max_words: int = 3) -> list[str]:
    """Ambil n-gram yang tepinya bukan stopword, tanpa melewati tanda baca."""
    found = []
    for chunk in re.split(r"[^\w\s-]+", text):
        words = chunk.split()
        for n in range(1, max_words + 1):
            for i in range(len(words) - n + 1):
                gram = words[i : i + n]
                if gram[0] in all_stopwords or gram[-1] in all_stopwords or len(gram[0]) < 3:
                    continue
                found.append(" ".join(gram))
    return list(dict.fromkeys(found))


print(
    candidate_phrases("program mbg di sekolah kami belum berjalan dengan baik, dapur masih kotor")[
        :8
    ]
)

['sekolah', 'berjalan', 'baik', 'berjalan dengan baik', 'dapur', 'kotor', 'dapur masih kotor']


## 6. Validasi

Metrik: precision, recall, dan F1 pada `TOP_K` frasa teratas, dengan pencocokan persis setelah huruf kecil.
Pencocokan persis cukup keras, jadi nilai absolutnya wajar rendah. Yang penting perbandingan antar metode.

In [10]:
def f1_at_k(
    predictions: list[list[str]], golds: list[set[str]], k: int = TOP_K
) -> dict[str, float]:
    tp = pred_total = gold_total = 0
    for pred, gold in zip(predictions, golds):
        top = pred[:k]
        tp += len(set(top) & gold)
        pred_total += len(top)
        gold_total += len(gold)
    precision = tp / max(pred_total, 1)
    recall = tp / max(gold_total, 1)
    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(2 * precision * recall / max(precision + recall, 1e-9), 4),
    }


assert f1_at_k([["a", "b"]], [{"a"}], 2) == {"precision": 0.5, "recall": 1.0, "f1": 0.6667}
results: list[dict] = []


def record(name: str, predictions: list[list[str]]) -> None:
    results.append({"metode": name, **f1_at_k(predictions, eval_gold)})
    print(results[-1])

## 7. Baseline

TF-IDF: frasa yang sering di dokumen ini tapi jarang di dokumen lain. YAKE: fitur statistik satu dokumen
saja, tidak butuh korpus. Keduanya tanpa model dari luar.

In [11]:
import yake  # noqa: E402
from sklearn.feature_extraction.text import TfidfVectorizer as PhraseTfidf  # noqa: E402

with stage("baseline"):
    phrase_tfidf = PhraseTfidf(
        ngram_range=(1, 3), min_df=1, sublinear_tf=True, token_pattern=r"(?u)\b[\w-]+\b"
    )
    corpus_matrix = phrase_tfidf.fit_transform(eval_texts)
    idf = dict(zip(phrase_tfidf.get_feature_names_out(), phrase_tfidf.idf_))

    def tfidf_rank(text: str) -> list[str]:
        candidates = candidate_phrases(text)
        return sorted(candidates, key=lambda c: -text.count(c) * idf.get(c, 0) * len(c.split()))

    tfidf_pred = [tfidf_rank(t) for t in eval_texts]
    extractor = yake.KeywordExtractor(lan="id", n=3, top=TOP_K * 3, stopwords=all_stopwords)
    yake_pred = [[kw.lower() for kw, _ in extractor.extract_keywords(t)] for t in eval_texts]
record("tfidf", tfidf_pred)
record("yake", yake_pred)

[waktu] baseline: 1 detik
{'metode': 'tfidf', 'precision': 0.0877, 'recall': 0.1088, 'f1': 0.0971}
{'metode': 'yake', 'precision': 0.1292, 'recall': 0.16, 'f1': 0.143}


## 8. Model Utama

**Embedding (gaya KeyBERT).** Frasa yang embedding-nya paling dekat dengan embedding dokumen dianggap paling
mewakili isi.

In [12]:
def embedding_rank_all(
    texts: list[str],
) -> tuple[list[list[str]], list[np.ndarray], list[list[str]]]:
    """Kembalikan peringkat frasa, skor, dan kandidat untuk tiap dokumen."""
    doc_vectors = e5_embed(texts, prefix="passage: ")
    candidates = [candidate_phrases(t)[:150] for t in texts]
    unique = sorted({c for row in candidates for c in row})
    lookup = dict(zip(unique, e5_embed(unique, prefix="query: ")))
    ranked, stored = [], []
    for doc_vector, row in zip(doc_vectors, candidates):
        if not row:
            ranked.append([])
            stored.append(np.zeros((0, doc_vectors.shape[1])))
            continue
        vectors = np.vstack([lookup[c] for c in row])
        ranked.append([row[i] for i in np.argsort(-(vectors @ doc_vector))])
        stored.append(vectors)
    return ranked, [doc_vectors, stored], candidates


with stage("embedding"):
    if USE_PRETRAINED:
        embed_pred, embed_store, embed_candidates = embedding_rank_all(eval_texts)
    else:
        embed_pred = tfidf_pred
record("embedding", embed_pred)

[waktu] embedding: 44 detik
{'metode': 'embedding', 'precision': 0.1482, 'recall': 0.1839, 'f1': 0.1642}


## 9. Post-processing

Dua langkah: buang frasa yang terkandung di frasa lain yang peringkatnya lebih tinggi, dan MMR supaya hasilnya
beragam, tidak lima variasi dari frasa yang sama.

In [13]:
def drop_overlaps(phrases: list[str]) -> list[str]:
    kept: list[str] = []
    for phrase in phrases:
        if not any(phrase in other or other in phrase for other in kept):
            kept.append(phrase)
    return kept


def mmr_rank(doc_vector, vectors, row, k=TOP_K) -> list[str]:
    """Maximal Marginal Relevance: dekat dengan dokumen, jauh dari frasa terpilih."""
    relevance = vectors @ doc_vector
    chosen = [int(np.argmax(relevance))]
    while len(chosen) < min(k, len(row)):
        redundancy = (vectors @ vectors[chosen].T).max(1)
        score = (1 - MMR_DIVERSITY) * relevance - MMR_DIVERSITY * redundancy
        score[chosen] = -np.inf
        chosen.append(int(np.argmax(score)))
    return [row[i] for i in chosen]


record("tfidf_buang_tumpang_tindih", [drop_overlaps(p) for p in tfidf_pred])
record("yake_buang_tumpang_tindih", [drop_overlaps(p) for p in yake_pred])
if USE_PRETRAINED:
    doc_vectors, stored = embed_store
    mmr_pred = [
        mmr_rank(d, v, c) if c else [] for d, v, c in zip(doc_vectors, stored, embed_candidates)
    ]
    record("embedding_mmr", mmr_pred)

{'metode': 'tfidf_buang_tumpang_tindih', 'precision': 0.0741, 'recall': 0.0774, 'f1': 0.0757}
{'metode': 'yake_buang_tumpang_tindih', 'precision': 0.0969, 'recall': 0.1003, 'f1': 0.0985}
{'metode': 'embedding_mmr', 'precision': 0.2443, 'recall': 0.3031, 'f1': 0.2706}


## 10. Post-analysis

**Tabel perbandingan.**

In [14]:
result_table = pd.DataFrame(results).sort_values("f1", ascending=False)
result_table.to_csv(RESULT_DIR / "perbandingan.csv", index=False)
result_table

,metode,precision,recall,f1
5,embedding_mmr,0.2443,0.3031,0.2706
2,embedding,0.1482,0.1839,0.1642
1,yake,0.1292,0.1600,0.1430
4,yake_buang_tumpang_tindih,0.0969,0.1003,0.0985
0,tfidf,0.0877,0.1088,0.0971
3,tfidf_buang_tumpang_tindih,0.0741,0.0774,0.0757


**Contoh hasil** dibanding jawaban, untuk menilai apakah frasa yang "salah" sebenarnya masih masuk akal.

In [15]:
best_method = result_table.iloc[0]["metode"]
named = {
    "tfidf": tfidf_pred,
    "yake": yake_pred,
    "embedding": embed_pred,
    "tfidf_buang_tumpang_tindih": [drop_overlaps(p) for p in tfidf_pred],
    "yake_buang_tumpang_tindih": [drop_overlaps(p) for p in yake_pred],
}
if USE_PRETRAINED:
    named["embedding_mmr"] = mmr_pred
best_pred = named[best_method]
pd.DataFrame(
    [
        {
            "teks": eval_texts[i][:80],
            "jawaban": sorted(eval_gold[i])[:4],
            "prediksi": best_pred[i][:4],
        }
        for i in range(5)
    ]
)

,teks,jawaban,prediksi
0,setelah melalui proses telepon yang panjang tutup sudah kartu kredit bca ribet,"[kartu kredit bca, melalui, proses telepon, ribet]","[kartu kredit bca, proses telepon, ribet, panjang tutup]"
1,@halobca saya mencoba mengakses menu m-bca saya namun saya mendapat respons fasi,"[@halobca, mendapat respons, mengakses, menu m-bca]","[mobile banking terblokir, halobca saya mencoba, mengakses menu m-bca, respons fasilitas]"
2,hanya saya atau @halobca klikbca bisnis memang error,"[@halobca, error, klikbca bisnis]","[halobca klikbca bisnis, error, bisnis memang error, halobca]"
3,teller bank bca ini menanyakan kabar saya sopan sekali,[sopan],"[kabar saya sopan, teller bank bca, bca ini menanyakan, sekali]"
4,bca senantiasa menjaga rahasia data customer,"[bca, data customer, menjaga, rahasia]","[bca senantiasa menjaga, data customer, rahasia data, bca]"


**Penerapan ke komentar MBG.** Frasa diberi skor TF-IDF di seluruh korpus, lalu dijumlahkan, sehingga yang
muncul adalah isu utama seluruh data dan bukan kata kunci satu komentar.

In [16]:
words_only = target["clean"].map(lambda t: " ".join(w for w in re.findall(r"[a-z]{3,}", t)))
corpus_tfidf = PhraseTfidf(
    ngram_range=(1, 3), min_df=5, max_df=0.2, sublinear_tf=True, token_pattern=r"\b[a-z]{3,}\b"
)
matrix = corpus_tfidf.fit_transform(words_only)
scores = np.asarray(matrix.sum(0)).ravel()
order = np.argsort(-scores)
names = corpus_tfidf.get_feature_names_out()
mbg_phrases = [names[i] for i in order if not (set(names[i].split()) & all_stopwords)][:400]
mbg_top = drop_overlaps(mbg_phrases)[:25]
pd.DataFrame({"frasa": mbg_top}).to_csv(RESULT_DIR / "isu_mbg.csv", index=False)
print(", ".join(mbg_top))

prof, gema, korupsi, rakyat, anak, indonesia, negara, pemerintah, sekolah, baik, bagus, presiden, setuju, uang, prabowo, dapur, mantap, harga, sppg, data, koruptor, masyarakat, kasih, petani, lapangan


## 11. Keluaran

File untuk dikumpulkan dan ringkasan run.

In [17]:
submission = pd.DataFrame(
    {"id": range(len(best_pred)), "kata_kunci": ["; ".join(p[:TOP_K]) for p in best_pred]}
)
submission.to_csv(RESULT_DIR / "submission.csv", index=False)
run_summary(
    "kata_kunci", len(target), {r["metode"]: r["f1"] for r in results}, needs_gpu=USE_PRETRAINED
)

RINGKASAN_RUN task=kata_kunci mode=penuh n_rows=5000 total_detik=58 gpu=True
{
 "tfidf": 0.0971,
 "yake": 0.143,
 "embedding": 0.1642,
 "tfidf_buang_tumpang_tindih": 0.0757,
 "yake_buang_tumpang_tindih": 0.0985,
 "embedding_mmr": 0.2706
}


**Poin untuk presentasi**

- Embedding + MMR terbaik: F1 0,2706, sedangkan YAKE 0,143 dan TF-IDF 0,0971.
- MMR menaikkan embedding dari 0,1642 ke 0,2706. Membuang tumpang tindih saja justru menurunkan YAKE dan TF-IDF.
- Tanpa label, tetap pakai metode yang terbukti di KEPS, lalu tunjukkan hasilnya pada data soal.
- Kata kunci MBG yang muncul: korupsi, rakyat, anak, sekolah, dapur, harga, SPPG, petani.